In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error, 
    mean_squared_error,
    root_mean_squared_error, 
    mean_absolute_percentage_error,
    make_scorer
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

## Dataset import

In [2]:
df = pd.read_csv("../data/processed/bc_clean.csv")
df = df.drop(["Unnamed: 0"], axis=1)

In [3]:
X = df.drop(["price"], axis=1)
Y = df[["price"]]

### Linear models

#### Train test split

In [4]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, shuffle=True
)

#### Pipeline

The class below will be applied to columns containing values = 0 after the imputation. It'll be used inside a column transformer

In [5]:
class StandardizeIgnoreZero(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, Y=None):
        X = np.asarray(X, dtype=float).copy()
        self.means_ = []
        self.stds_ = []

        for i in range(X.shape[1]):
            mask = X[:, i] != 0
            mean = X[mask, i].mean()
            std = X[mask, i].std()
    
            self.means_.append(mean)
            self.stds_.append(std)
        
        return self

    def transform(self, X, Y=None):
        X = np.asarray(X, dtype=float).copy()
        for i in range(X.shape[1]):
            mask = X[:, i] != 0
            X[mask, i] = (X[mask, i] - self.means_[i]) / self.stds_[i]
        return X

In [9]:
numeric_col = ["latitude", "longitude", "property-beds", "property-baths"]
num_col_ignore_zero = ["Acreage", "Property Tax", "Square Footage"]

ct = ColumnTransformer(
    transformers = [
        ("std_ignore_zeros", StandardizeIgnoreZero(), num_col_ignore_zero),
        ("std", StandardScaler(), numeric_col)
    ],
    remainder = "passthrough"
)

In [10]:
pipe = Pipeline([
    ("col_transform", ct),
    ("model", LinearRegression())
])

In [11]:
param_grid = [
    {
        "model" : [LinearRegression()]
    }, 
    {
        "model" : [Ridge()],
        "model__alpha" : [0.001, 0.01, 0.1, 1, 10],
        "model__max_iter" : [3000]
    },
    {
        "model" : [Lasso()],
        "model__alpha" : [0.001, 0.01, 0.1, 1, 10],
        "model__max_iter" : [3000]

    },
    {
        "model" : [ElasticNet()],
        "model__alpha" : [0.001, 0.01, 0.1, 1, 10],
        "model__l1_ratio" : [0.1, 0.5, 0.9],
        "model__max_iter" : [3000]

    },  
    {
        "model" : [KNeighborsRegressor()],
        "model__n_neighbors" : np.arange(5, 15, 2),
        "model__p" : [1, 2],
        "model__weights" : ["uniform", "distance"]
    }
]

In [12]:
scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False),
}

In [13]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid = param_grid,
    scoring = scoring,
    refit = "neg_rmse",
    cv = KFold(5),
    n_jobs = 10,
    verbose = 4
)

In [90]:
grid.fit(X_train, Y_train)

Fitting 5 folds for each of 46 candidates, totalling 230 fits


/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.210e+14, tolerance: 4.966e+12
  model = cd_fast.enet_coordinate_descent(
/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.285e+14, tolerance: 5.058e+12
  model = cd_fast.enet_coordinate_descent(
/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [LinearRegression()]}, {'model': [Ridge()], 'model__alpha': [0.001, 0.01, ...], 'model__max_iter': [3000]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'neg_mae': make_scorer(m...hod='predict'), 'neg_mape': make_scorer(m...hod='predict'), 'neg_rmse': make_scorer(r...hod='predict'), 'r2_score': make_scorer(r...hod='predict')}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'neg_rmse'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default 

In [95]:
grid.best_params_

{'model': KNeighborsRegressor(),
 'model__n_neighbors': np.int64(7),
 'model__p': 1,
 'model__weights': 'distance'}

In [96]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("R²:", r2_score(Y_test, y_pred))
print("MAE:", mean_absolute_error(Y_test, y_pred))
print("RMSE:", root_mean_squared_error(Y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(Y_test, y_pred))

R²: 0.6894406351107201
MAE: 405551.0159064908
RMSE: 964569.2747073396
MAPE: 0.23733243470514692


In [84]:
df['price'].describe()

count    2.252300e+04
mean     1.578931e+06
std      1.814173e+06
min      3.880000e+04
25%      6.890000e+05
50%      1.058000e+06
75%      1.799900e+06
max      5.880000e+07
Name: price, dtype: float64

### Tree based models

In [88]:
pipe_trees = Pipeline([
    ('model', RandomForestRegressor())
])

trees_grid = {
    "model__n_estimators": [100, 500],
    "model__max_depth": [None, 10, 30],
    "model__min_samples_split": [2, 7],
    "model__min_samples_leaf": [1, 3],
    "model__max_features": ["sqrt", "log2"]
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False),
}

In [89]:
grid_trees = GridSearchCV(
    estimator=pipe_trees,
    param_grid = trees_grid,
    scoring = scoring,
    refit = "neg_rmse",
    cv = KFold(3),
    n_jobs = 11,
    verbose = 3
)

In [90]:
grid_trees.fit(X_train, Y_train.to_numpy().ravel())

Fitting 3 folds for each of 48 candidates, totalling 144 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 3], 'model__min_samples_split': [2, 7], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'neg_mae': make_scorer(m...hod='predict'), 'neg_mape': make_scorer(m...hod='predict'), 'neg_mse': make_scorer(m...hod='predict'), 'neg_rmse': make_scorer(r...hod='predict'), ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",11
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'neg_rmse'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... ve

In [91]:
grid_trees.best_params_

{'model__max_depth': None,
 'model__max_features': 'sqrt',
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__n_estimators': 500}

In [94]:
best_model_trees = grid_trees.best_estimator_

y_pred = best_model_trees.predict(X_test)

print("R²:", r2_score(Y_test, y_pred))
print("MAE:", mean_absolute_error(Y_test, y_pred))
print("RMSE:", root_mean_squared_error(Y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(Y_test, y_pred))

R²: 0.8040535652961822
MAE: 296950.21029954404
RMSE: 766178.2015889843
MAPE: 0.18344453778282094
